In [16]:
# import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [17]:
train =  pd.read_parquet("../../data/03_model_ready/ratings_train.parquet")
test =  pd.read_parquet("../../data/03_model_ready/ratings_test.parquet")
dfUser = pd.read_parquet("../../data/02_processed/ratings_integrity.parquet")
dfUser = dfUser.sort_values(['userId', 'timestamp'])
# def particion(ratingsUser):
#     global train, test 
#     n = len(ratingsUser)
#     p = round(n*0.7)
#     trainPar = ratingsUser.head(p)
#     testPar = ratingsUser.tail(n - p)
#     train = pd.concat([train, trainPar], ignore_index=True)
#     test = pd.concat([test, testPar], ignore_index=True)

# usuarios = dfUser['userId'].unique()
# for i in usuarios:
#     subconjunto = dfUser[dfUser['userId']==i]
#     particion(subconjunto)

In [18]:
df = pd.read_parquet("PLN.parquet").sort_values('movieId')
df.head()

,movieId,title,genres,tag,overview,join,texto_TFIDF
0,1,Toy Story,adventure animation children comedy fantasy,pixar pixar fun,led by woody andys toys live happily in his ro...,adventure animation children comedy fantasy pi...,adventure animation child comedy fantasy pixar...
1,2,Jumanji,adventure children fantasy,fantasy magic board game robin williams game,when siblings judy and peter discover an encha...,adventure children fantasy fantasy magic board...,adventure child fantasy fantasy magic board ga...
2,3,Grumpier Old Men,comedy romance,moldy old,a family wedding reignites the ancient feud be...,comedy romance moldy old A family wedding reig...,comedy romance moldy old family wedding reigni...
3,4,Waiting to Exhale,comedy drama romance,,cheated on mistreated and stepped on the women...,"comedy drama romance Cheated on, mistreated a...","comedy drama romance cheat , mistreat step , w..."
4,5,Father of the Bride Part II,comedy,pregnancy remake,just when george banks has recovered from his ...,comedy pregnancy remake Just when George Banks...,comedy pregnancy remake george bank recover da...


In [19]:
#corpus de documentos 
corpus = df['texto_TFIDF'].tolist()

# objeto 
vectorizer = TfidfVectorizer( ) # max_features=5000,min_df=5,max_df=0.8

# fit aprende el vocabulario completo y transform vectoriza cada documento
# se tiene una coordenada por cada palabra del vocabulario
X = vectorizer.fit_transform(corpus)

# muestra las palabras que forman el vocabulario
vectorizer.get_feature_names_out()

array(['00', '000', '007', ..., 'žižek', 'ʻohana', 'сhatterer'],
      shape=(23292,), dtype=object)

In [20]:
# df_tfidf_content = pd.DataFrame(
#     X.toarray(),
#     columns=vectorizer.get_feature_names_out(),
#     index=df['movieId']
# )
# df_tfidf_content
# datos = pd.merge(df_tfidf_content, train, on = 'movieId', how = 'right')

# generos = vectorizer.get_feature_names_out().tolist()

# # hay peliculas que no tienen genero asociado
# datos[generos] = datos[generos].fillna(0)

# #usando la media ponderada por pesos(ratings). usuario es DF
# def agregacion(usuario):
#     vectores= usuario[generos].values
#     pesos    = usuario['rating'].values
#     return np.average(vectores, axis = 0, weights=pesos) # 0 es para que sea por columna

# df_tfidf_BasedProfile = datos.groupby('userId').apply(agregacion).reset_index() 

# df_tfidf_BasedProfile.columns=['userId', 'BasedProfile']
# df_tfidf_BasedProfile
# no entra en memoria

In [21]:
from scipy import sparse
import numpy as np

corpus = vectorizer.get_feature_names_out()

# X ya es sparse (la salida de TfidfVectorizer)
# Crear un mapping de movieId a índice en X
movie_to_idx = {mid: i for i, mid in enumerate(df['movieId'])}

perfiles = []
for user_id, group in train.groupby('userId'):
    indices = [movie_to_idx[m] for m in group['movieId'] if m in movie_to_idx]
    if indices:
        vectores = X[indices]  # sigue siendo sparse, no consume memoria
        pesos = group.loc[group['movieId'].isin(movie_to_idx), 'rating'].values
        # Media ponderada: sum(vector_i * peso_i) / sum(pesos)
        perfil = np.array((vectores.T @ pesos) / pesos.sum()).flatten()
    else:
        perfil = np.zeros(len(corpus))
    perfiles.append({'userId': user_id, 'BasedProfile': perfil})

df_tfidf_BasedProfile = pd.DataFrame(perfiles)

In [22]:
len(corpus)

23292

In [25]:
def evaluacion(userId, k=10, t=4):
    vector_BP = df_tfidf_BasedProfile[df_tfidf_BasedProfile['userId' ]==userId]['BasedProfile'].values[0]
    similitudes = cosine_similarity([vector_BP], X)[0]
    datosUserId =  pd.DataFrame({
    'movieId' : df['movieId'],
    'title' :df['title'],
    'similitud' : similitudes
    })
    pelisVistas = train[train['userId']==userId]['movieId'].tolist()
    pelisRec = datosUserId[~datosUserId['movieId'].isin(pelisVistas)].sort_values('similitud',ascending =False).head(k)
    datos = pd.merge(test[test['userId']==userId], pelisRec, on="movieId", how = 'right')
    interseccion = datos[~datos['rating'].isna()] #peliculas vistas y que han sido recomendadas

    nRelevantes = (interseccion['rating'] >= t).sum()
    nTest = (test[test["userId"]==userId]['rating'] >= t).sum() #pelis test relevantes
    precisionk = nRelevantes/k
    if nTest !=0:
        recalk = nRelevantes/nTest
    else:
        recalk = 0
    if recalk + precisionk != 0:
        F1k = 2*recalk*precisionk/(recalk + precisionk)
    else:
        F1k = 0
  
    resultados =  pd.DataFrame({
        'userId' : [userId],
        'precisionK': [precisionk],
        'recalK':[recalk],
        'F1K':[F1k]
        })
    return resultados
    
def resultadosUmbral(t, p=100):
    listaUs = dfUser['userId'].unique()
    evaluacionTFIDF = pd.DataFrame()
    for i in listaUs:
        evaluacionTFIDF = pd.concat([evaluacionTFIDF,evaluacion(i,t=t)], ignore_index=True)

    solucion = evaluacionTFIDF[['precisionK', 'recalK','F1K']].mean()*p 
    return solucion


In [26]:
resultadosUmbral(3.5)

precisionK    0.098361
recalK        0.037652
F1K           0.038398
dtype: float64

In [ ]:
def evaluacionMOD(userId, k=10, t=4):
    vector_BP = df_tfidf_BasedProfile[df_tfidf_BasedProfile['userId' ]==userId]['BasedProfile'].values[0]
    similitudes = cosine_similarity([vector_BP], X)[0]
    datosUserId =  pd.DataFrame({
    'movieId' : df['movieId'],
    'title' :df['title'],
    'similitud' : similitudes
    })
    pelisVistas = train[train['userId']==userId]['movieId'].tolist()
    pelisRec = datosUserId[~datosUserId['movieId'].isin(pelisVistas)].sort_values('similitud',ascending =False).head(k)
    datos = pd.merge(test[test['userId']==userId], pelisRec, on="movieId", how = 'right')
    interseccion = datos[~datos['rating'].isna()] #peliculas vistas y que han sido recomendadas

    nRelevantes = (interseccion['rating'] >= t).sum()
    nTest = (test[test["userId"]==userId]['rating'] >= t).sum() #pelis test relevantes
    precisionk = nRelevantes/k
    if nTest !=0:
        recalk = nRelevantes/nTest
    else:
        recalk = 0
    if recalk + precisionk != 0:
        F1k = 2*recalk*precisionk/(recalk + precisionk)
    else:
        F1k = 0
  
    resultados =  pd.DataFrame({
        'userId' : [userId],
        'precisionK': [precisionk],
        'recalK':[recalk],
        'F1K':[F1k]
        })
    return resultados
    
def resultadosUmbralMOD(t, p=100):
    listaUs = dfUser['userId'].unique()
    evaluacionTFIDF = pd.DataFrame()
    for i in listaUs:
        evaluacionTFIDF = pd.concat([evaluacionTFIDF,evaluacionMOD(i,t=t)], ignore_index=True)

    solucion = evaluacionTFIDF[['precisionK', 'recalK','F1K']].mean()*p 
    return solucion